In [22]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [23]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_final12_pca.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [24]:
# 데이터를 읽어온다.
train_df = pd.read_csv('(C_D_NotCD)_pca(20)_train_final.csv')
test_df = pd.read_csv('(C_D_NotCD)_pca(20)_test.csv')

display(train_df)
display(test_df)

,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,...,p14,p15,p16,p17,p18,p19,p20,Group,기준년월,ID
0,-150783.968438,-56082.020946,51037.941736,-27832.077192,16764.902567,-6722.901448,8017.228716,-1589.899745,-1903.356424,-619.245196,...,695.702007,-2118.462504,-1639.881651,920.069726,511.718859,-1122.756368,123.505008,D,201807,TRAIN_000000
1,-29393.379986,9938.404430,115081.083211,-20135.707035,-5857.585601,-10019.036113,4556.178547,2057.356990,3765.857653,7995.277636,...,-1951.968365,1409.617899,-5706.378995,-1780.678516,-1422.397326,635.862388,644.018418,C,201807,TRAIN_000002
2,-142353.565715,-64154.223712,93221.199407,-23492.918679,20369.557438,3236.070104,15395.461901,-5636.916532,2905.972770,1382.190089,...,1017.672162,-261.330256,-2476.748714,2372.291183,296.830258,-653.930605,-127.836243,D,201807,TRAIN_000003
3,340845.132838,117915.936314,-93128.551494,19111.767843,-46991.887054,-4445.300905,13447.551193,-16563.630643,46086.915040,-5327.911980,...,-5701.039627,-12319.162335,-1307.383944,-2781.299443,-930.195183,-1520.976434,-729.562992,C,201807,TRAIN_000008
4,-97905.472256,78320.279698,6494.134578,-4516.307322,-11954.702338,-9874.741005,13430.785456,290.018531,3971.263032,-258.815143,...,476.545295,7168.964576,10412.741513,356.362678,-2980.915622,808.135107,-1931.182976,D,201807,TRAIN_000010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477943,-29150.170034,22181.385835,9281.710609,17381.517971,12538.906825,-8956.998938,3810.385934,-4792.506364,796.826643,4882.946518,...,-6904.505576,3599.247784,-1534.464112,-832.748652,2307.587176,-934.073365,202.979713,D,201812,TRAIN_399979
477944,-17513.333798,-70697.580000,53946.608206,18710.117945,4641.402693,-8406.709125,179.866941,2628.154881,-20.116104,3924.466640,...,-1575.192315,5945.342511,-4745.918684,-1407.972710,1362.822015,-1292.967815,403.869160,C,201812,TRAIN_399987
477945,47482.389701,-48215.021320,-46932.195110,18507.641154,-23785.809085,-6300.144546,-17444.229352,883.859491,-14881.342817,-7770.090738,...,954.730644,-7979.744714,170.807025,3883.683243,-680.865443,370.019957,-719.823786,C,201812,TRAIN_399993
477946,25163.340661,-58.369545,-2920.886645,-25589.429101,24855.350199,1761.901868,-4908.108591,17155.243996,-3882.382805,5421.548716,...,-678.111723,650.241867,2302.705505,-4382.323726,-2808.337380,466.888353,307.112222,D,201812,TRAIN_399996


,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,...,p12,p13,p14,p15,p16,p17,p18,p19,p20,ID
0,-13705.067440,46467.691559,15942.555179,50207.425732,13279.118384,-10540.445197,8275.998167,597.220989,-2948.674740,-439.575976,...,1706.048258,-965.838040,-161.714809,-719.020089,-2860.625905,3313.634852,342.230181,-284.058508,-1063.562006,TEST_00002
1,-157243.879719,-13302.476700,-30608.699050,-10141.276718,-20191.567610,16734.803593,3865.040703,2766.081509,-7436.949339,15593.166090,...,-12124.200787,11666.408502,-4085.345752,752.390808,-1908.040291,-2157.231466,-2163.578964,157.956699,178.894712,TEST_00010
2,-48997.752561,-17050.828103,-8785.969908,12658.477978,976.577391,-4781.244178,-1387.027484,3497.772702,-3101.396425,-10387.737288,...,3833.353734,-924.614911,6633.882026,1840.229396,-4320.036898,2559.579926,2611.515095,1951.500122,-453.318890,TEST_00012
3,52617.221072,-117970.778109,-63093.266659,-13703.021001,-525.569384,-20709.360313,-11409.284360,16661.488453,4419.949193,-1665.345793,...,-9062.662800,1465.081008,-557.481777,-979.536980,-1589.199659,-1228.275578,1990.620854,3028.806924,-720.985493,TEST_00016
4,-150893.648894,23186.461190,-25302.261145,-980.885632,-19613.383003,27205.851461,8992.013284,-103.473319,-6977.553648,-11877.372912,...,-15599.675321,732.953370,4740.377962,648.517284,-539.828000,-1710.122138,-865.854859,-827.977316,-66.660295,TEST_00032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96493,5534.292187,-48302.147655,-35833.803086,20868.002514,-16374.674953,-5379.702512,-11563.581200,-3282.599440,2630.263551,1120.416128,...,2287.983091,1015.485091,160.905425,5216.652475,3256.013247,-3489.050837,1456.987510,541.779349,-382.434722,TEST_99957
96494,-32783.504355,-53443.942457,-33865.260054,9599.980027,16651.123902,-15976.790692,-13476.342166,1921.854194,-4305.693411,5777.082963,...,-5417.860671,3484.478509,-2302.378322,-1166.886929,-226.575257,1158.781200,-789.949760,-2876.429032,2098.222234,TEST_99961
96495,390074.729741,96464.046501,-105134.555049,-88208.920785,-2426.837122,19983.301997,13348.602388,41972.099226,-16046.998083,4965.756830,...,9229.586083,1084.252045,-2342.334199,12283.910664,-11720.708318,-2909.645085,1386.512242,1992.611980,-131.153143,TEST_99982
96496,-46298.872570,13607.202059,118027.253642,-1223.211427,-13979.589030,-10674.375726,27371.224836,9007.675594,12317.703831,8117.270221,...,1389.377297,-382.730890,-2926.906637,3525.806139,-4855.369976,-508.454855,3470.659567,-894.316036,-765.165143,TEST_99994


In [25]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,...,p14,p15,p16,p17,p18,p19,p20,Group,기준년월,ID
0,-150783.968438,-56082.020946,51037.941736,-27832.077192,16764.902567,-6722.901448,8017.228716,-1589.899745,-1903.356424,-619.245196,...,695.702007,-2118.462504,-1639.881651,920.069726,511.718859,-1122.756368,123.505008,D,201807.0,TRAIN_000000
1,-29393.379986,9938.404430,115081.083211,-20135.707035,-5857.585601,-10019.036113,4556.178547,2057.356990,3765.857653,7995.277636,...,-1951.968365,1409.617899,-5706.378995,-1780.678516,-1422.397326,635.862388,644.018418,C,201807.0,TRAIN_000002
2,-142353.565715,-64154.223712,93221.199407,-23492.918679,20369.557438,3236.070104,15395.461901,-5636.916532,2905.972770,1382.190089,...,1017.672162,-261.330256,-2476.748714,2372.291183,296.830258,-653.930605,-127.836243,D,201807.0,TRAIN_000003
3,340845.132838,117915.936314,-93128.551494,19111.767843,-46991.887054,-4445.300905,13447.551193,-16563.630643,46086.915040,-5327.911980,...,-5701.039627,-12319.162335,-1307.383944,-2781.299443,-930.195183,-1520.976434,-729.562992,C,201807.0,TRAIN_000008
4,-97905.472256,78320.279698,6494.134578,-4516.307322,-11954.702338,-9874.741005,13430.785456,290.018531,3971.263032,-258.815143,...,476.545295,7168.964576,10412.741513,356.362678,-2980.915622,808.135107,-1931.182976,D,201807.0,TRAIN_000010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574441,5534.292187,-48302.147655,-35833.803086,20868.002514,-16374.674953,-5379.702512,-11563.581200,-3282.599440,2630.263551,1120.416128,...,160.905425,5216.652475,3256.013247,-3489.050837,1456.987510,541.779349,-382.434722,NaN,NaN,TEST_99957
574442,-32783.504355,-53443.942457,-33865.260054,9599.980027,16651.123902,-15976.790692,-13476.342166,1921.854194,-4305.693411,5777.082963,...,-2302.378322,-1166.886929,-226.575257,1158.781200,-789.949760,-2876.429032,2098.222234,NaN,NaN,TEST_99961
574443,390074.729741,96464.046501,-105134.555049,-88208.920785,-2426.837122,19983.301997,13348.602388,41972.099226,-16046.998083,4965.756830,...,-2342.334199,12283.910664,-11720.708318,-2909.645085,1386.512242,1992.611980,-131.153143,NaN,NaN,TEST_99982
574444,-46298.872570,13607.202059,118027.253642,-1223.211427,-13979.589030,-10674.375726,27371.224836,9007.675594,12317.703831,8117.270221,...,-2926.906637,3525.806139,-4855.369976,-508.454855,3470.659567,-894.316036,-765.165143,NaN,NaN,TEST_99994


In [27]:
all_X=all_df.drop(columns=['ID', 'Group', '기준년월'])
all_y = all_df['Group']

In [28]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_X)

,copy,True
,with_mean,True
,with_std,True


In [29]:
target1=pd.read_parquet(r'data/train/1.회원정보/201807_train_.parquet')
target2=pd.read_parquet(r'data/train/1.회원정보/201808_train_.parquet')
target3=pd.read_parquet(r'data/train/1.회원정보/201809_train_.parquet')
target4=pd.read_parquet(r'data/train/1.회원정보/201810_train_.parquet')
target5=pd.read_parquet(r'data/train/1.회원정보/201811_train_.parquet')
target6=pd.read_parquet(r'data/train/1.회원정보/201812_train_.parquet')

In [30]:
tg_df = pd.concat([
    target1['Segment'],
    target2['Segment'],
    target3['Segment'],
    target4['Segment'],
    target5['Segment'],
    target6['Segment']
])

tg_df = tg_df.reset_index(drop=True).to_frame(name='Segment')
tg_df = tg_df[tg_df['Segment'] != 'E']

tg_df['Segment'] = tg_df['Segment'].apply(
    lambda x: x if x in ['C', 'D'] else 'not CD'
)
tg_df

,Segment
0,D
2,C
3,D
8,C
10,D
...,...
2399979,D
2399987,C
2399993,C
2399996,D


In [31]:
X = train_df.drop(columns=['Group','기준년월','ID'])
y = train_df['Group']

In [32]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
y = le.fit_transform(y)

In [33]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-1.07510059, -0.65037877,  0.7895155 , ...,  0.22171764,
        -0.53950375,  0.11036885],
       [-0.24560077,  0.13788076,  1.79655077, ..., -0.63656346,
         0.31589029,  0.5674252 ],
       [-1.01749301, -0.7467579 ,  1.45281882, ...,  0.12635893,
        -0.3114664 , -0.11033078],
       ...,
       [ 0.27971538, -0.5564497 , -0.75099892, ..., -0.30750215,
         0.18658425, -0.63014768],
       [ 0.12720235,  0.01852297, -0.05895073, ..., -1.25158657,
         0.23370114,  0.27159206],
       [-0.74670443,  0.07986894, -0.16901529, ..., -0.36375053,
         0.29226698,  0.74612504]])

In [34]:
train_X = X2
train_y = y

In [35]:
scaler_columns = X.columns.tolist()
scaler_columns

['p1',
 'p2',
 'p3',
 'p4',
 'p5',
 'p6',
 'p7',
 'p8',
 'p9',
 'p10',
 'p11',
 'p12',
 'p13',
 'p14',
 'p15',
 'p16',
 'p17',
 'p18',
 'p19',
 'p20']

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [36]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=3, verbose=-1)

kfold = KFold(n_splits=10, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

평균 f1 Score : 0.8029890279149555


In [37]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.8075


In [38]:
# RandomForest
rf_basic_model = RandomForestClassifier()
# 교차 검증을 수행한다
r1 = cross_val_score(rf_basic_model, train_X, train_y, scoring='f1_micro', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("RandomForest Basic")

print(f'평균 f1 Score : {r1.mean()}')

KeyboardInterrupt: 

In [39]:
df = pd.DataFrame({
    'Model': model_name_list,
    'f1 score': f1_score_list
})
df = df.dropna()

In [40]:
df

,Model,f1 score
0,LGBMClassifier,0.802989
1,XGBClassifier,0.807536


In [41]:
final_model=model6.fit(train_X, train_y)

In [42]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(le, fp)

print('저장완료')

저장완료
